# Premier League Validation: Unchanged Cascade on New Data

Phase 1 — Data Inspection, part 5. Objective: run notebook 04's approved cascade *without logic changes* on Premier League 2015/16 (StatsBomb competition 2, season 27, 380 matches; Transfermarkt GB1, 2015-07-01 to 2016-06-30) and check whether the yield and failure modes match La Liga.

Structural difference under test: Premier League club naming (StatsBomb short forms such as `Arsenal` vs Transfermarkt suffixed forms such as `Arsenal FC`) should be absorbed by the existing containment gate with zero new map entries. Any mismatch that is genuinely new proposes a `CLUB_MAP` addition — recorded, not silently applied.

All matching logic is imported from `src.join`; this notebook contains no duplicate matching code.

## Cell 1 — Setup, scope, and module imports

`sys.path` gains the repo root so `src.join` imports from the notebook directory. Season constants mirror notebook 04 with the league sides swapped to England.

In [1]:
import json, os, sys
import pandas as pd
import numpy as np

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
from src.join import CLUB_MAP, NICKNAMES, clubs_agree, normalize_name, resolve_sample

TM_DIR = os.path.join(ROOT, 'data', 'transfermarkt')
SB_DATA_DIR = os.path.join(ROOT, 'data', 'statsbomb', 'data')
SB_COMP, SB_SEASON = 2, 27
TM_COMP = 'GB1'
SEASON_START, SEASON_END = '2015-07-01', '2016-06-30'

print('root:', ROOT)
print('module map entries:', len(CLUB_MAP), '| nicknames:', len(NICKNAMES))


root: c:\Users\User\Development\football-recruitment-intelligence
module map entries: 4 | nicknames: 18


## Cell 2 — Base tables (both sides)

Same construction as notebook 04: StatsBomb lineup files reduced to distinct player-team pairs; Transfermarkt season-windowed appearances resolved to club names. Normalization reuses `normalize_name` from the module.

In [2]:
ms = json.load(open(os.path.join(SB_DATA_DIR, 'matches', str(SB_COMP), f'{SB_SEASON}.json'), encoding='utf-8'))
print('matches:', len(ms))
sb_rows = []
for m in ms:
    fp = os.path.join(SB_DATA_DIR, 'lineups', f"{m['match_id']}.json")
    for t in json.load(open(fp, encoding='utf-8')):
        for p in t['lineup']:
            sb_rows.append((p['player_name'], t['team_name']))
df_sb = pd.DataFrame(sb_rows, columns=['sb_player', 'sb_team']).drop_duplicates()
df_sb['sb_key'] = df_sb['sb_player'].apply(normalize_name)

app = pd.read_csv(os.path.join(TM_DIR, 'appearances.csv'),
                  usecols=['player_id', 'player_name', 'player_club_id', 'date', 'competition_id'],
                  parse_dates=['date'])
es = app[(app['competition_id'] == TM_COMP) & (app['date'] >= SEASON_START) & (app['date'] < '2016-07-01')]
clubs = pd.read_csv(os.path.join(TM_DIR, 'clubs.csv'), usecols=['club_id', 'name'])
df_tm = (es[['player_id', 'player_name', 'player_club_id']].drop_duplicates()
           .merge(clubs, left_on='player_club_id', right_on='club_id')
           .rename(columns={'name': 'tm_club'})[['player_id', 'player_name', 'tm_club']])
df_tm['tm_key'] = df_tm['player_name'].apply(normalize_name)
print('distinct SB pairs:', len(df_sb), '| players:', df_sb['sb_player'].nunique())
print('distinct TM pairs:', len(df_tm), '| players:', df_tm['player_name'].nunique())


matches: 380
distinct SB pairs: 664 | players: 649
distinct TM pairs: 560 | players: 549


## Cell 3 — Unchanged cascade and manual-review table

`resolve_sample` runs exactly as approved: token-subset, club gate, disambiguation cascade, nicknames, fuzzy. Anything still ambiguous prints with date of birth and position for human sign-off rather than being resolved silently.

In [3]:
final, still = resolve_sample(df_sb, df_tm)
print('FINAL matches:', len(final), '/', df_sb['sb_player'].nunique(), 'SB players')
print(final['method'].value_counts().to_dict())
print('still ambiguous:', len(still))
if len(still):
    bio = pd.read_csv(os.path.join(TM_DIR, 'players.csv'),
                      usecols=['player_id', 'date_of_birth', 'position', 'height_in_cm'])
    pairs = df_sb.merge(df_tm, left_on='sb_key', right_on='tm_key', how='inner')
    manual = pairs[pairs['sb_player'].isin(still)] \
        [['sb_player', 'sb_team', 'player_id', 'player_name', 'tm_club']] \
        .drop_duplicates().merge(bio, on='player_id', how='left')
    print(manual.to_string())


FINAL matches: 539 / 649 SB players
{'exact': 384, 'token': 137, 'nickname': 15, 'fuzzy': 3}
still ambiguous: 0


## Cell 4 — New-map-entry audit

Rebuilds candidate pairs to list every SB-team to TM-club mismatch. Entries genuinely new to the Premier League (same club, different naming) propose `CLUB_MAP` additions; transfer-driven and false-positive mismatches must stay rejected. Nothing is added to the module from this cell — proposals go through review first.

In [4]:
from src.join.match import candidate_pairs
pairs = candidate_pairs(df_sb, df_tm)
mm = pairs.loc[~pairs['club_ok'], ['sb_team', 'tm_club']].drop_duplicates().sort_values(['sb_team', 'tm_club'])
print('mismatched team pairs:', len(mm))
print(mm.to_string())


mismatched team pairs: 28
                     sb_team               tm_club
89169        AFC Bournemouth        Crystal Palace
134522       AFC Bournemouth          Norwich City
22709            Aston Villa          Liverpool FC
28826            Aston Villa  West Bromwich Albion
371658               Chelsea       West Ham United
331884        Crystal Palace       AFC Bournemouth
265382        Crystal Palace          Norwich City
297283               Everton          Norwich City
99959         Leicester City          Swansea City
171686             Liverpool            Chelsea FC
361528             Liverpool        Southampton FC
342207      Newcastle United          Swansea City
343652      Newcastle United     Tottenham Hotspur
271512          Norwich City       AFC Bournemouth
202463          Norwich City        Crystal Palace
200623          Norwich City            Everton FC
205157          Norwich City       West Ham United
82236            Southampton          Liverpool FC
31789

## Cell 5 — Valuation attachment and final tally

Latest valuation on or before season end for each resolved player; unmatched players reported, not hidden. Compare against La Liga (442/601, 74%): a similar yield with no new map entries means the method generalizes; divergence localizes to league-specific naming or squad churn.

In [5]:
final['split_season'] = final['sb_player'].isin(
    df_sb.groupby('sb_player')['sb_team'].nunique()[lambda s: s > 1].index)
val = pd.read_csv(os.path.join(TM_DIR, 'player_valuations.csv'), parse_dates=['date'])
val = val[val['date'] <= SEASON_END].sort_values('date').drop_duplicates('player_id', keep='last')
final = final.merge(val[['player_id', 'date', 'market_value_in_eur']], on='player_id', how='left')
print('with valuation:', final['market_value_in_eur'].notna().sum(), '/', len(final))
print('split-season rows:', final['split_season'].sum())
print('unmatched SB players:', df_sb['sb_player'].nunique() - final['sb_player'].nunique())
print()
print(final.head(10).to_string())


with valuation: 515 / 539
split-season rows: 13
unmatched SB players: 110

             sb_player          sb_team  player_id        player_name          tm_club method  split_season       date  market_value_in_eur
0      Aaron Cresswell  West Ham United      92571    Aaron Cresswell  West Ham United  exact         False 2016-02-09           10000000.0
1         Aaron Lennon          Everton      14221       Aaron Lennon       Everton FC  exact         False 2016-02-09            6000000.0
2         Aaron Ramsey          Arsenal      50057       Aaron Ramsey       Arsenal FC  exact         False 2016-06-01           30000000.0
3    Abdul Rahman Baba          Chelsea     224884  Abdul Rahman Baba       Chelsea FC  exact         False 2016-02-09           13000000.0
4   Adam David Lallana        Liverpool      43530       Adam Lallana     Liverpool FC  token         False 2016-06-01           19000000.0
5        Adam Federici  AFC Bournemouth       4233      Adam Federici  AFC Bournemout

## Next steps

1. Review any still-ambiguous rows and the Cell 4 mismatch list.
2. Approve or reject proposed `CLUB_MAP` additions (module change + test, if any).
3. Scale the approved cascade to all overlapping league-seasons.
4. Engineer per-90 features on the joined player-season rows, then design Postgres.